In [2]:
import os
import pandas as pd

from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# Load environment variables from .env file
load_dotenv()

# Get database credentials from environment variables
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')

# Create database connection string
DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# Create database engine
engine = create_engine(DATABASE_URL)

print("Environment variables loaded successfully")
print(f"Connected to: {DB_NAME} on {DB_HOST}:{DB_PORT}")

Environment variables loaded successfully
Connected to: humanitarian_db on localhost:5432


In [3]:
# List of tables to preview
tables = [
    'cleaned_data.education_facilities',
    'cleaned_data.health_facilities',
    'cleaned_data.health_facility_type',
    'cleaned_data.jiaf_south_sudan_2026_clean',
    'cleaned_data.population_estimates_2024_clean',
    'gis.country_boundary',
    'gis.county_boundary',
    'gis.state_boundary'
]

dataframes = {}

for table in tables:    
    query = f"SELECT * FROM {table};"
    # Extract just the table name (everything after the dot)
    table_name = table.split('.')[-1]
    dataframes[table_name] = pd.read_sql(query, engine)
    print(f"Loaded {table_name}")
    
print("All tables loaded successfully")

Loaded education_facilities
Loaded health_facilities
Loaded health_facility_type
Loaded jiaf_south_sudan_2026_clean
Loaded population_estimates_2024_clean
Loaded country_boundary
Loaded county_boundary
Loaded state_boundary
All tables loaded successfully


In [5]:
# Structural Profiling for all loaded tables

structural_profile = {}

for name, df in dataframes.items():
    profile = {
        'row_count': len(df),
        'column_count': len(df.columns),
        'column_names': df.columns.tolist(),
        'dtypes': df.dtypes.to_dict(),
        'duplicate_rows': df.duplicated().sum(),
        'duplicate_columns': sum([df.columns.duplicated().sum()])  # Checks for duplicate column names
    }
    
    # Identify potential primary key candidates (columns with all unique values)
    pk_candidates = []
    for col in df.columns:
        if df[col].nunique() == len(df):
            pk_candidates.append(col)
    profile['pk_candidates'] = pk_candidates
    
    structural_profile[name] = profile
    
    # Print summary
    print(f"\n{'='*60}")
    print(f"TABLE: {name}")
    print(f"{'='*60}")
    print(f"Rows: {profile['row_count']:,}")
    print(f"Columns: {profile['column_count']}")
    print(f"Duplicate Rows: {profile['duplicate_rows']}")
    print(f"Potential PKs: {profile['pk_candidates']}")
    print(f"Column Names: {', '.join(profile['column_names'])}")

print("\nStructural profiling complete for all tables")


TABLE: education_facilities
Rows: 438
Columns: 11
Duplicate Rows: 0
Potential PKs: ['id', 'geometry']
Column Names: id, name, amenity, adm1_pcode, adm1_name, adm2_pcode, adm2_name, adm3_pcode, adm3_name, name_latin, geometry

TABLE: health_facilities
Rows: 1,988
Columns: 10
Duplicate Rows: 0
Potential PKs: []
Column Names: state, state_code, county, county_code, payam, payam_code, facility_name, latitude, longitude, missing_coordinates

TABLE: health_facility_type
Rows: 1,513
Columns: 12
Duplicate Rows: 0
Potential PKs: []
Column Names: State, State_Code, County, County_Code, Payam, Payam_Code , Facility_Name, Facility_type, Facilities_Code, Latitude, Longitude, missing_coordinates

TABLE: jiaf_south_sudan_2026_clean
Rows: 239
Columns: 17
Duplicate Rows: 0
Potential PKs: []
Column Names: col_1, col_2, col_3, col_4, col_5, col_6, sectoral_pin_number, col_7, col_8, col_9, col_10, col_11, col_12, col_13, col_14, col_15, col_16

TABLE: population_estimates_2024_clean
Rows: 80
Columns: 21


In [6]:
# Completeness Profiling - Missing values, blanks, and placeholders

import numpy as np

# Common placeholder strings that often mean "missing"
PLACEHOLDERS = ['unknown', 'n/a', 'none', 'nan', '-', 'not applicable', 'null', 'na', 'n.a.']

completeness_profile = {}

for name, df in dataframes.items():
    total_rows = len(df)
    col_stats = []
    
    for col in df.columns:
        series = df[col]
        
        # Count NaNs and None
        null_count = series.isna().sum()
        
        # For string columns, count blanks (empty strings) and whitespace-only
        if series.dtype == 'object':
            # Convert to string to safely check
            str_series = series.astype(str)
            blank_count = (str_series == '').sum()
            whitespace_only = (str_series.str.strip() == '').sum() - blank_count  # non-empty but all spaces
            # Count placeholders (case-insensitive)
            lower_series = str_series.str.lower().str.strip()
            placeholder_count = lower_series.isin(PLACEHOLDERS).sum()
        else:
            blank_count = 0
            whitespace_only = 0
            placeholder_count = 0
        
        # For numeric columns, NaN is the only missing indicator typically
        filled_count = total_rows - null_count - blank_count
        
        col_stats.append({
            'column': col,
            'dtype': str(series.dtype),
            'total': total_rows,
            'null_nan': null_count,
            'null_nan_pct': round(100 * null_count / total_rows, 1),
            'blank': blank_count,
            'blank_pct': round(100 * blank_count / total_rows, 1),
            'whitespace_only': whitespace_only,
            'placeholder_count': placeholder_count,
            'placeholder_pct': round(100 * placeholder_count / total_rows, 1),
            'filled': filled_count,
            'filled_pct': round(100 * filled_count / total_rows, 1)
        })
    
    completeness_profile[name] = col_stats
    
    # Print summary
    print(f"\n{'='*70}")
    print(f"TABLE: {name}")
    print(f"{'='*70}")
    print(f"{'Column':<30} {'Dtype':<12} {'Null%':>7} {'Blank%':>7} {'Placeh%':>7} {'Filled%':>7}")
    print("-"*70)
    for stat in col_stats:
        print(f"{stat['column']:<30} {stat['dtype']:<12} {stat['null_nan_pct']:>6.1f}% {stat['blank_pct']:>6.1f}% {stat['placeholder_pct']:>6.1f}% {stat['filled_pct']:>6.1f}%")
    print("-"*70)

print("\nCompleteness profiling complete for all tables")


TABLE: education_facilities
Column                         Dtype          Null%  Blank% Placeh% Filled%
----------------------------------------------------------------------
id                             object          0.0%    0.0%    0.0%  100.0%
name                           object          0.0%    0.0%    0.0%  100.0%
amenity                        object          3.9%    0.0%    3.9%   96.1%
adm1_pcode                     object          0.0%    0.0%    0.0%  100.0%
adm1_name                      object          0.0%    0.0%    0.0%  100.0%
adm2_pcode                     object          0.0%    0.0%    0.0%  100.0%
adm2_name                      object          0.0%    0.0%    0.0%  100.0%
adm3_pcode                     object          0.0%    0.0%    0.0%  100.0%
adm3_name                      object          0.0%    0.0%    0.0%  100.0%
name_latin                     object          0.0%    0.0%    0.0%  100.0%
geometry                       object          0.0%    0.0%    0

In [7]:
# Uniqueness Profiling - Distinct counts and duplicate detection

uniqueness_profile = {}

for name, df in dataframes.items():
    total_rows = len(df)
    col_stats = []
    
    for col in df.columns:
        distinct_count = df[col].nunique(dropna=True)  # Exclude NaN from count
        # A column is unique if distinct count equals total row count (ignoring NaN)
        is_unique = (distinct_count == total_rows)
        # Count rows where the value appears more than once (excluding NaN)
        value_counts = df[col].value_counts(dropna=True)
        duplicate_values = value_counts[value_counts > 1]
        duplicate_row_count = duplicate_values.sum()  # rows that have a duplicated value
        
        col_stats.append({
            'column': col,
            'distinct': distinct_count,
            'total_rows': total_rows,
            'unique_pct': round(100 * distinct_count / total_rows, 1),
            'is_unique': is_unique,
            'duplicate_values_count': len(duplicate_values),
            'rows_with_duplicates': duplicate_row_count,
            # Sample duplicated values (up to 5)
            'dup_examples': duplicate_values.head(5).index.tolist() if not duplicate_values.empty else []
        })
    
    uniqueness_profile[name] = col_stats
    
    # Print summary
    print(f"\n{'='*90}")
    print(f"TABLE: {name}")
    print(f"{'='*90}")
    print(f"{'Column':<30} {'Distinct':>10} {'Unique%':>8} {'Rows w/ Dup':>12} {'Dup Values':>12} {'Sample Dups'}")
    print("-"*90)
    for stat in col_stats:
        dup_info = f"{stat['rows_with_duplicates']:,}" if stat['rows_with_duplicates'] > 0 else "-"
        dup_vals = f"{stat['duplicate_values_count']:,}" if stat['duplicate_values_count'] > 0 else "-"
        samples = ", ".join([str(x) for x in stat['dup_examples']]) if stat['dup_examples'] else "-"
        print(f"{stat['column']:<30} {stat['distinct']:>10,} {stat['unique_pct']:>7.1f}% {dup_info:>12} {dup_vals:>12} {samples[:60]}")
    print("-"*90)

print("\nUniqueness profiling complete for all tables")


TABLE: education_facilities
Column                           Distinct  Unique%  Rows w/ Dup   Dup Values Sample Dups
------------------------------------------------------------------------------------------
id                                    438   100.0%            -            - -
name                                  383    87.4%           84           29 St Augustine Secondary School, St Kizito Primary School, Rom
amenity                                 5     1.1%          420            4 school, college, university, kindergarten
adm1_pcode                             10     2.3%          438           10 SS01, SS10, SS07, SS02, SS09
adm1_name                              10     2.3%          438           10 Central Equatoria, Western Equatoria, Upper Nile, Eastern Eq
adm2_pcode                             52    11.9%          424           38 SS0101, SS1005, SS0106, SS0102, SS0903
adm2_name                              52    11.9%          424           38 Juba, Mundri West,

In [9]:
# Text Profiling - compact summary (stores full details in text_profile dict)

import re

text_profile = {}

for name, df in dataframes.items():
    text_cols = df.select_dtypes(include=['object']).columns.tolist()
    if not text_cols:
        continue
    
    table_summary = []
    for col in text_cols:
        series = df[col].dropna()
        total = len(series)
        if total == 0:
            table_summary.append([col, 0, '-', '-', '-', '-', '-', '-', '-'])
            continue
        
        s = series.astype(str)
        lead = s.str.match(r'^\s').sum()
        trail = s.str.match(r'.*\s$').sum()
        multi_sp = s.str.contains(r'\s{2,}').sum()
        mixed_c = (s.str.contains(r'[A-Z]') & s.str.contains(r'[a-z]')).sum()
        non_ascii = s.str.contains(r'[^\x00-\x7F]').sum()
        max_l = s.str.len().max()
        
        # Flag if any issues exist
        issues = []
        if lead > 0: issues.append('lead_ws')
        if trail > 0: issues.append('trail_ws')
        if multi_sp > 0: issues.append('multi_sp')
        if mixed_c > 0: issues.append('mixed_case')
        if non_ascii > 0: issues.append('non_ascii')
        if max_l > 100: issues.append('long_vals')
        looks_num = pd.to_numeric(series, errors='coerce').notna().all()
        if looks_num: issues.append('numeric_text')
        
        table_summary.append([col, total, lead, trail, multi_sp, mixed_c, non_ascii, max_l, ', '.join(issues) if issues else 'Clean'])
    
    print(f"\n{'='*90}")
    print(f"TABLE: {name} - TEXT PROFILING")
    print(f"{'='*90}")
    print(f"{'Column':<28} {'Rows':>6} {'LeadWS':>7} {'TrailWS':>7} {'MultiSp':>7} {'MixCase':>7} {'NonASCII':>8} {'MaxLen':>6}   {'Issues'}")
    print('-' * 90)
    for row in table_summary:
        print(f"{row[0]:<28} {row[1]:>6} {row[2]:>7} {row[3]:>7} {row[4]:>7} {row[5]:>7} {row[6]:>8} {row[7]:>6}   {row[8]}")
    print('-' * 90)
    
    # Still store detailed results for later inspection (top values, etc.)
    detailed = {}
    for col in text_cols:
        series = df[col].dropna()
        detailed[col] = {
            'total': len(series),
            'top10': series.value_counts().head(10).to_dict() if len(series) > 0 else {}
        }
    text_profile[name] = detailed

print("\nText profiling complete (compact). Full top-values stored in text_profile dict.")


TABLE: education_facilities - TEXT PROFILING
Column                         Rows  LeadWS TrailWS MultiSp MixCase NonASCII MaxLen   Issues
------------------------------------------------------------------------------------------
id                              438       0       0       0       0        0     17   Clean
name                            438       0       0       0     420        1     73   mixed_case, non_ascii
amenity                         421       0       0       0       0        0     16   Clean
adm1_pcode                      438       0       0       0       0        0      4   Clean
adm1_name                       438       0       0       0     438        0     23   mixed_case
adm2_pcode                      438       0       0       0       0        0      6   Clean
adm2_name                       438       0       0       0     438        0     14   mixed_case
adm3_pcode                      438       0       0       0       0        0      8   Clean
adm3_nam

In [10]:
# Numerical Profiling - summary stats, negatives, zeros, outliers, impossible values

numerical_profile = {}

for name, df in dataframes.items():
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not num_cols:
        print(f"\nTABLE: {name} - No numeric columns, skipping.")
        continue
    
    print(f"\n{'='*110}")
    print(f"TABLE: {name} - NUMERICAL PROFILING")
    print(f"{'='*110}")
    print(f"{'Column':<25} {'Min':>12} {'Max':>12} {'Mean':>12} {'StdDev':>12} {'Neg':>6} {'Zero':>6} {'Outlier?':>10}")
    print('-' * 110)
    
    table_stats = {}
    for col in num_cols:
        series = df[col].dropna()
        if len(series) == 0:
            continue
        
        min_v = series.min()
        max_v = series.max()
        mean_v = series.mean()
        std_v = series.std()
        neg = (series < 0).sum()
        zero = (series == 0).sum()
        
        # Basic outlier detection: values beyond 3 std deviations from mean
        if std_v > 0:
            z_scores = (series - mean_v) / std_v
            outliers = (np.abs(z_scores) > 3).sum()
        else:
            outliers = 0
        
        table_stats[col] = {
            'min': min_v,
            'max': max_v,
            'mean': mean_v,
            'std': std_v,
            'negatives': neg,
            'zeros': zero,
            'outliers_3std': outliers
        }
        
        outlier_flag = f"{outliers} rows" if outliers > 0 else "No"
        print(f"{col:<25} {min_v:>12,.2f} {max_v:>12,.2f} {mean_v:>12,.2f} {std_v:>12,.2f} {neg:>6} {zero:>6} {outlier_flag:>10}")
    print('-' * 110)
    numerical_profile[name] = table_stats

print("\nNumerical profiling complete for all tables")


TABLE: education_facilities - No numeric columns, skipping.

TABLE: health_facilities - NUMERICAL PROFILING
Column                             Min          Max         Mean       StdDev    Neg   Zero   Outlier?
--------------------------------------------------------------------------------------------------------------
latitude                          3.54        12.17         6.96         2.11      0      0         No
longitude                        24.82        35.07        30.27         2.10      0      0         No
--------------------------------------------------------------------------------------------------------------

TABLE: health_facility_type - NUMERICAL PROFILING
Column                             Min          Max         Mean       StdDev    Neg   Zero   Outlier?
--------------------------------------------------------------------------------------------------------------
Facilities_Code           71,010,101.00 93,080,602.00 83,067,135.57 7,827,524.58      0      0 

In [12]:
# Categorical Profiling - compact, long values truncated

categorical_profile = {}

for name, df in dataframes.items():
    cat_cols = df.select_dtypes(include=['object']).columns.tolist()
    if not cat_cols:
        print(f"\nTABLE: {name} - No categorical columns, skipping.")
        continue
    
    print(f"\n{'='*110}")
    print(f"TABLE: {name} - CATEGORICAL PROFILING")
    print(f"{'='*110}")
    print(f"{'Column':<28} {'#Distinct':>9} {'/Total':>7} {'Top 3 Values (Count, truncated)'}                            {'Singletons':>10} {'Flag'}")
    print('-' * 110)
    
    table_details = {}
    for col in cat_cols:
        series = df[col].dropna()
        total = len(series)
        if total == 0:
            continue
        
        vc = series.value_counts()
        distinct = len(vc)
        singletons = (vc == 1).sum()
        
        # Format top3 with truncation
        top3_items = []
        for val, cnt in list(vc.head(3).items()):
            val_str = str(val)
            if len(val_str) > 40:
                val_str = val_str[:37] + "..."
            top3_items.append(f"'{val_str}' ({cnt})")
        top_str = ", ".join(top3_items)
        
        # Flags
        flags = []
        if pd.to_numeric(series, errors='coerce').notna().all():
            flags.append("NumericText")
        if distinct > 50:
            flags.append("HighCard")
        if singletons > 0:
            flags.append(f"S:{singletons}")
        flag_str = ", ".join(flags)
        
        table_details[col] = {
            'distinct': distinct,
            'total': total,
            'full_freq': vc.to_dict()
        }
        
        print(f"{col:<28} {distinct:>9,} {distinct/total:>6.1%} {top_str:<55} {singletons:>10} {flag_str}")
    print('-' * 110)
    categorical_profile[name] = table_details

print("\nCategorical profiling complete (full frequencies stored in categorical_profile dict).")


TABLE: education_facilities - CATEGORICAL PROFILING
Column                       #Distinct  /Total Top 3 Values (Count, truncated)                            Singletons Flag
--------------------------------------------------------------------------------------------------------------
id                                 438 100.0% 'way/974164633' (1), 'way/974164631' (1), 'way/974164630' (1)        438 HighCard, S:438
name                               383  87.4% 'St Augustine Secondary School' (9), 'St Kizito Primary School' (9), 'Roman Catholic Primary School' (7)        354 HighCard, S:354
amenity                              5   1.2% 'school' (393), 'college' (13), 'university' (10)                1 S:1
adm1_pcode                          10   2.3% 'SS01' (144), 'SS10' (71), 'SS07' (55)                           0 
adm1_name                           10   2.3% 'Central Equatoria' (144), 'Western Equatoria' (71), 'Upper Nile' (55)          0 
adm2_pcode                          52  1

In [15]:
# GIS Profiling - coordinates, geometry, spatial plausibility (fixed detection)

# South Sudan approximate bounds (degrees)
SS_LAT_MIN, SS_LAT_MAX = 3.0, 13.0
SS_LON_MIN, SS_LON_MAX = 23.0, 36.0

def is_lat_col(col_name):
    """Check if column name suggests latitude (avoid false positives like 'county')."""
    name = col_name.lower().strip()
    # Match 'lat', 'latitude', or standalone 'y' / 'ycoord'
    if 'lat' in name or 'latitude' in name:
        return True
    if name in ('y', 'ycoord', 'y_coord'):
        return True
    return False

def is_lon_col(col_name):
    """Check if column name suggests longitude."""
    name = col_name.lower().strip()
    if 'lon' in name or 'lng' in name or 'long' in name or 'longitude' in name:
        return True
    if name in ('x', 'xcoord', 'x_coord'):
        return True
    return False

gis_profile = {}

for name, df in dataframes.items():
    geo_cols = [c for c in df.columns if c.lower() in ('geometry', 'geom', 'wkt', 'the_geom')]
    lat_cols = [c for c in df.columns if is_lat_col(c)]
    lon_cols = [c for c in df.columns if is_lon_col(c)]
    
    if not geo_cols and not (lat_cols and lon_cols):
        print(f"\nTABLE: {name} - No GIS columns detected, skipping.")
        continue
    
    print(f"\n{'='*110}")
    print(f"TABLE: {name} - GIS PROFILING")
    print(f"{'='*110}")
    
    table_result = {'geometry_cols': {}, 'coord_cols': {}}
    
    # --- Geometry columns ---
    for col in geo_cols:
        series = df[col].astype(str)
        total = len(series)
        null_count = series.isna().sum()
        blank_count = (series.str.strip() == '').sum()
        empty_geom_count = series.str.contains(r'^EMPTY$', case=False).sum() if total > 0 else 0
        filled = total - null_count - blank_count
        
        # Determine geometry types (first word of WKT)
        type_counts = {}
        if filled > 0:
            types = series.str.extract(r'^([A-Za-z]+)')[0].value_counts()
            type_counts = types.to_dict()
        
        table_result['geometry_cols'][col] = {
            'total': total,
            'null': null_count,
            'blank': blank_count,
            'empty': empty_geom_count,
            'filled': filled,
            'geom_types': type_counts
        }
        
        types_str = ', '.join([f'{k}:{v}' for k,v in type_counts.items()]) if type_counts else '-'
        print(f"  GEOM col: {col}")
        print(f"    Rows: {total} | Null: {null_count} | Blank: {blank_count} | EmptyGeom: {empty_geom_count}")
        print(f"    Filled: {filled} | Geometry types: {types_str}")
    
    # --- Coordinate columns ---
    # Pairing: if numbers of lat and lon cols differ, warn and pair by order; otherwise skip bad conversions
    if lat_cols and lon_cols:
        # Attempt to pair them (assume same order, which may not be perfect)
        for lat_col, lon_col in zip(lat_cols, lon_cols):
            try:
                lat_series = pd.to_numeric(df[lat_col], errors='coerce')
                lon_series = pd.to_numeric(df[lon_col], errors='coerce')
            except Exception as e:
                print(f"  COORD cols: {lat_col}, {lon_col} - Cannot convert to numeric: {e}")
                continue
            
            lat_null = lat_series.isna().sum()
            lon_null = lon_series.isna().sum()
            lat_valid = lat_series.dropna()
            lon_valid = lon_series.dropna()
            
            lat_min, lat_max = (lat_valid.min(), lat_valid.max()) if len(lat_valid) > 0 else (None, None)
            lon_min, lon_max = (lon_valid.min(), lon_valid.max()) if len(lon_valid) > 0 else (None, None)
            
            # Plausibility flags
            flags = []
            if lat_max is not None and (lat_max > 90 or lat_min < -90):
                flags.append('LAT_OUT_OF_GLOBAL_RANGE')
            if lon_max is not None and (lon_max > 180 or lon_min < -180):
                flags.append('LON_OUT_OF_GLOBAL_RANGE')
            if lat_max is not None and lat_min is not None and (lat_max > SS_LAT_MAX or lat_min < SS_LAT_MIN):
                flags.append('LAT_OUTSIDE_SS')
            if lon_max is not None and lon_min is not None and (lon_max > SS_LON_MAX or lon_min < SS_LON_MIN):
                flags.append('LON_OUTSIDE_SS')
            
            zero_zero = 0
            if len(lat_valid) > 0 and len(lon_valid) > 0:
                zero_zero = ((lat_valid == 0) & (lon_valid == 0)).sum()
            if zero_zero > 0:
                flags.append(f'ZERO_ZERO:{zero_zero}')
            
            # Check swapped lat/lon
            swap_flag = False
            if lat_max is not None and lat_min is not None and lon_max is not None and lon_min is not None:
                if abs(lat_max - lat_min) > 40 and abs(lon_max - lon_min) < 15:
                    swap_flag = True
                    flags.append('LIKELY_SWAPPED')
                elif lat_max > 60 and lat_min < -60:
                    swap_flag = True
                    flags.append('LIKELY_SWAPPED')
            
            coord_info = {
                'lat_col': lat_col,
                'lon_col': lon_col,
                'lat_null': lat_null,
                'lon_null': lon_null,
                'lat_range': (lat_min, lat_max),
                'lon_range': (lon_min, lon_max),
                'zero_zero': zero_zero,
                'flags': flags
            }
            table_result['coord_cols'][f"{lat_col},{lon_col}"] = coord_info
            
            flag_str = ', '.join(flags) if flags else 'OK'
            # Safely format ranges
            lat_range_str = f"{lat_min:.4f} to {lat_max:.4f}" if lat_min is not None else "No data"
            lon_range_str = f"{lon_min:.4f} to {lon_max:.4f}" if lon_min is not None else "No data"
            print(f"\n  COORD cols: {lat_col}, {lon_col}")
            print(f"    Lat range: {lat_range_str} (nulls: {lat_null})")
            print(f"    Lon range: {lon_range_str} (nulls: {lon_null})")
            print(f"    Zero-zero points: {zero_zero}")
            print(f"    Flags: {flag_str}")
    
    gis_profile[name] = table_result

print("\nGIS profiling complete (details stored in gis_profile dict).")


TABLE: education_facilities - GIS PROFILING
  GEOM col: geometry
    Rows: 438 | Null: 0 | Blank: 0 | EmptyGeom: 0
    Filled: 438 | Geometry types: -

TABLE: health_facilities - GIS PROFILING

  COORD cols: latitude, longitude
    Lat range: 3.5416 to 12.1695 (nulls: 113)
    Lon range: 24.8158 to 35.0749 (nulls: 113)
    Zero-zero points: 0
    Flags: OK

TABLE: health_facility_type - GIS PROFILING

  COORD cols: Latitude, Longitude
    Lat range: 3.5416 to 12.1695 (nulls: 225)
    Lon range: 24.8158 to 35.0749 (nulls: 225)
    Zero-zero points: 0
    Flags: OK

TABLE: jiaf_south_sudan_2026_clean - No GIS columns detected, skipping.

TABLE: population_estimates_2024_clean - No GIS columns detected, skipping.

TABLE: country_boundary - GIS PROFILING
  GEOM col: geometry
    Rows: 1 | Null: 0 | Blank: 0 | EmptyGeom: 0
    Filled: 1 | Geometry types: -

  COORD cols: center_lat, center_lon
    Lat range: 7.8620 to 7.8620 (nulls: 0)
    Lon range: 29.1178 to 29.1178 (nulls: 0)
    Zero-

In [16]:
# Relational Profiling - foreign keys, lookups, admin hierarchy

relational_profile = []

def check_fk(source_df, source_col, target_df, target_col, description):
    """Check that values in source_col exist in target_col."""
    if source_col not in source_df.columns:
        return f"SKIP: {source_col} not in source"
    if target_col not in target_df.columns:
        return f"SKIP: {target_col} not in target"
    src_vals = source_df[source_col].dropna().unique()
    tgt_vals = set(target_df[target_col].dropna().unique())
    missing = [v for v in src_vals if v not in tgt_vals]
    if missing:
        return f"FAIL: {len(missing)} values not found ({missing[:5]}{'...' if len(missing)>5 else ''})"
    return "OK"

def check_hierarchy(df, child_col, parent_col):
    """Check that each child value maps to exactly one parent value."""
    if child_col not in df.columns or parent_col not in df.columns:
        return f"SKIP: columns missing"
    mapping = df.dropna(subset=[child_col, parent_col]).groupby(child_col)[parent_col].nunique()
    ambiguous = mapping[mapping > 1]
    if len(ambiguous) > 0:
        return f"FAIL: {len(ambiguous)} {child_col} values map to multiple {parent_col}s (ex: {ambiguous.head(3).index.tolist()})"
    return "OK"

print(f"{'='*80}")
print("RELATIONAL PROFILING")
print(f"{'='*80}")

# --- Facility tables vs admin boundaries ---
for fac_table in ['education_facilities', 'health_facilities']:
    if fac_table in dataframes:
        df = dataframes[fac_table]
        print(f"\n--- {fac_table} ---")
        
        # adm1 check
        res = check_fk(df, 'adm1_pcode', dataframes['state_boundary'], 'adm1_pcode', 
                       f"{fac_table}.adm1_pcode -> state_boundary.adm1_pcode")
        print(f"  adm1_pcode in state_boundary: {res}")
        
        # adm2 check (county)
        res = check_fk(df, 'adm2_pcode', dataframes['county_boundary'], 'adm2_pcode',
                       f"{fac_table}.adm2_pcode -> county_boundary.adm2_pcode")
        print(f"  adm2_pcode in county_boundary: {res}")
        
        # For health facilities, check facility_type against lookup
        if fac_table == 'health_facilities':
            # Try common column names
            type_col = None
            for col in ['facility_type', 'type_code', 'type']:
                if col in df.columns:
                    type_col = col
                    break
            if type_col:
                res = check_fk(df, type_col, dataframes['health_facility_type'], 'type_code',
                               f"health_facilities.{type_col} -> health_facility_type.type_code")
                print(f"  facility_type in lookup: {res}")
            else:
                print("  facility_type lookup: SKIP (no type column found)")

# --- JIAF & population admin codes ---
for table_name in ['jiaf_south_sudan_2026_clean', 'population_estimates_2024_clean']:
    if table_name in dataframes:
        df = dataframes[table_name]
        print(f"\n--- {table_name} ---")
        for adm_level, boundary_table, code_col in [('adm1', 'state_boundary', 'adm1_pcode'),
                                                     ('adm2', 'county_boundary', 'adm2_pcode')]:
            adm_col = f"{adm_level}_pcode"
            if adm_col in df.columns:
                res = check_fk(df, adm_col, dataframes[boundary_table], code_col,
                               f"{table_name}.{adm_col} -> {boundary_table}.{code_col}")
                print(f"  {adm_col} in {boundary_table}: {res}")
            else:
                print(f"  {adm_col}: SKIP (column not found)")

# --- Administrative hierarchy checks ---
print("\n--- Admin hierarchy consistency ---")

# county_boundary: each adm2_pcode should map to exactly one adm1_pcode
if 'county_boundary' in dataframes:
    res = check_hierarchy(dataframes['county_boundary'], 'adm2_pcode', 'adm1_pcode')
    print(f"  county_boundary.adm2_pcode -> single adm1_pcode: {res}")
    
    # adm1_pcode in county_boundary should exist in state_boundary
    res = check_fk(dataframes['county_boundary'], 'adm1_pcode',
                   dataframes['state_boundary'], 'adm1_pcode',
                   "county_boundary.adm1_pcode -> state_boundary.adm1_pcode")
    print(f"  county_boundary.adm1_pcode in state_boundary: {res}")

# state_boundary: adm0_pcode should all be 'SS'
if 'state_boundary' in dataframes and 'adm0_pcode' in dataframes['state_boundary'].columns:
    states = dataframes['state_boundary']
    if not states['adm0_pcode'].eq('SS').all():
        bad = states[states['adm0_pcode'] != 'SS']['adm0_pcode'].unique().tolist()
        print(f"  state_boundary.adm0_pcode all 'SS': FAIL ({bad})")
    else:
        print(f"  state_boundary.adm0_pcode all 'SS': OK")

print("\nRelational profiling complete.")

RELATIONAL PROFILING

--- education_facilities ---
  adm1_pcode in state_boundary: OK
  adm2_pcode in county_boundary: OK

--- health_facilities ---
  adm1_pcode in state_boundary: SKIP: adm1_pcode not in source
  adm2_pcode in county_boundary: SKIP: adm2_pcode not in source
  facility_type lookup: SKIP (no type column found)

--- jiaf_south_sudan_2026_clean ---
  adm1_pcode: SKIP (column not found)
  adm2_pcode: SKIP (column not found)

--- population_estimates_2024_clean ---
  adm1_pcode: SKIP (column not found)
  adm2_pcode: SKIP (column not found)

--- Admin hierarchy consistency ---
  county_boundary.adm2_pcode -> single adm1_pcode: OK
  county_boundary.adm1_pcode in state_boundary: FAIL: 1 values not found (['SS00'])
  state_boundary.adm0_pcode all 'SS': OK

Relational profiling complete.


In [17]:
# Business Profiling - humanitarian domain rules

def find_col(df, candidates):
    """Return first column name from candidates that exists in df, else None."""
    for c in candidates:
        if c in df.columns:
            return c
    return None

def check_rule(description, condition, details=""):
    """Print a rule check result."""
    if condition:
        print(f"  {description}")
    else:
        print(f"  {description} {details}")

print(f"{'='*80}")
print("BUSINESS PROFILING")
print(f"{'='*80}")

# --- Population Estimates ---
if 'population_estimates_2024_clean' in dataframes:
    pop = dataframes['population_estimates_2024_clean']
    print("\n--- population_estimates_2024_clean ---")
    
    # Population > 0
    pop_col = find_col(pop, ['population', 'total_pop', 'pop_total', 'pop_est', 'est_population'])
    if pop_col:
        neg = (pop[pop_col] < 0).sum()
        zero = (pop[pop_col] == 0).sum()
        if neg > 0 or zero > 0:
            check_rule("Population > 0", False, f"({neg} negatives, {zero} zeros)")
        else:
            check_rule("Population > 0", True)
    
    # If children population exists, ensure children <= total
    child_col = find_col(pop, ['children', 'pop_children', 'u18', 'pop_under18', 'under18'])
    if pop_col and child_col:
        bad = (pop[child_col] > pop[pop_col]).sum()
        if bad > 0:
            check_rule("Children <= Total population", False, f"({bad} rows where children > total)")
        else:
            check_rule("Children <= Total population", True)

# --- JIAF ---
if 'jiaf_south_sudan_2026_clean' in dataframes:
    jiaf = dataframes['jiaf_south_sudan_2026_clean']
    print("\n--- jiaf_south_sudan_2026_clean ---")
    
    # Severity in 1-5 (or 1-4 depending on IPC/CH)
    sev_col = find_col(jiaf, ['severity', 'phase', 'ipc_phase', 'severity_class', 'classification'])
    if sev_col:
        valid_range = set(range(1,6))  # IPC 1-5
        actual_vals = set(jiaf[sev_col].dropna().unique())
        bad_vals = actual_vals - valid_range
        if bad_vals:
            check_rule(f"{sev_col} between 1-5", False, f"invalid values: {sorted(bad_vals)}")
        else:
            check_rule(f"{sev_col} between 1-5", True)
    
    # PIN <= Population (if both columns exist)
    pin_col = find_col(jiaf, ['pin', 'pop_in_need', 'population_in_need', 'p_i_n'])
    pop_col = find_col(jiaf, ['population', 'total_pop', 'pop_total', 'population_total'])
    if pin_col and pop_col:
        bad = (jiaf[pin_col] > jiaf[pop_col]).sum()
        if bad > 0:
            check_rule("PIN <= Population", False, f"({bad} rows where PIN > population)")
        else:
            check_rule("PIN <= Population", True)
    elif pin_col:
        print(f"  PIN column found ({pin_col}) but no population column to compare")
    elif pop_col:
        print(f"  Population column found ({pop_col}) but no PIN column")
    
    # Percentages (if columns like "pct_*" or "_percent" exist)
    pct_cols = [c for c in jiaf.columns if 'pct' in c.lower() or 'percent' in c.lower() or c.endswith('_share')]
    for col in pct_cols:
        try:
            vals = pd.to_numeric(jiaf[col], errors='coerce').dropna()
            if len(vals) > 0:
                min_v, max_v = vals.min(), vals.max()
                if min_v < 0 or max_v > 100:
                    check_rule(f"Percentage column '{col}' between 0-100", False, f"min={min_v:.1f}, max={max_v:.1f}")
                else:
                    check_rule(f"Percentage column '{col}' between 0-100", True)
        except:
            pass

# --- Health Facilities ---
if 'health_facilities' in dataframes:
    hf = dataframes['health_facilities']
    print("\n--- health_facilities ---")
    
    # Operational status
    status_col = find_col(hf, ['operational_status', 'status', 'facility_status', 'operational'])
    if status_col:
        valid_statuses = {'operational', 'functional', 'non-functional', 'closed', 'temporary', 'partially functional', 
                          'fully functional', 'open', 'yes', 'no', 'planned', 'under construction'}
        # Convert to lowercase set
        actual = set(hf[status_col].dropna().astype(str).str.lower().str.strip())
        unknown = actual - {s.lower() for s in valid_statuses}
        if unknown:
            check_rule(f"Facility status valid", False, f"unknown statuses: {sorted(unknown)[:5]}")
        else:
            check_rule(f"Facility status valid", True)
    
    # Coordinates already checked in GIS, but ensure essential facility fields non-null
    for essential_col in ['facility_name', 'adm2_pcode']:
        if essential_col in hf.columns:
            null_count = hf[essential_col].isna().sum()
            if null_count > 0:
                check_rule(f"{essential_col} not null", False, f"({null_count} nulls)")
            else:
                check_rule(f"{essential_col} not null", True)

# --- Education Facilities ---
if 'education_facilities' in dataframes:
    ef = dataframes['education_facilities']
    print("\n--- education_facilities ---")
    for essential_col in ['facility_name', 'adm2_pcode']:
        if essential_col in ef.columns:
            null_count = ef[essential_col].isna().sum()
            if null_count > 0:
                check_rule(f"{essential_col} not null", False, f"({null_count} nulls)")
            else:
                check_rule(f"{essential_col} not null", True)
    
    # Check if any school type column exists
    type_col = find_col(ef, ['school_type', 'facility_type', 'education_type'])
    if type_col:
        print(f"  Education type column '{type_col}' found – consider standardizing categories if needed")

# --- Admin boundary coverage ---
print("\n--- Admin Coverage ---")
if 'county_boundary' in dataframes and 'population_estimates_2024_clean' in dataframes:
    counties = set(dataframes['county_boundary']['adm2_pcode'].dropna())
    pop_counties = set(dataframes['population_estimates_2024_clean']['adm2_pcode'].dropna()) if 'adm2_pcode' in dataframes['population_estimates_2024_clean'].columns else set()
    missing_in_pop = counties - pop_counties
    extra_in_pop = pop_counties - counties
    if missing_in_pop:
        check_rule("All adm2 in county_boundary have population estimate", False, f"({len(missing_in_pop)} missing, e.g. {list(missing_in_pop)[:3]})")
    else:
        check_rule("All adm2 in county_boundary have population estimate", True)
    if extra_in_pop:
        check_rule("All population adm2 exist in county_boundary", False, f"({len(extra_in_pop)} extra, e.g. {list(extra_in_pop)[:3]})")
    else:
        check_rule("All population adm2 exist in county_boundary", True)

if 'county_boundary' in dataframes and 'jiaf_south_sudan_2026_clean' in dataframes:
    counties = set(dataframes['county_boundary']['adm2_pcode'].dropna())
    jiaf_counties = set(dataframes['jiaf_south_sudan_2026_clean']['adm2_pcode'].dropna()) if 'adm2_pcode' in dataframes['jiaf_south_sudan_2026_clean'].columns else set()
    missing_in_jiaf = counties - jiaf_counties
    extra_in_jiaf = jiaf_counties - counties
    if missing_in_jiaf:
        check_rule("All adm2 in county_boundary have JIAF entry", False, f"({len(missing_in_jiaf)} missing)")
    else:
        check_rule("All adm2 in county_boundary have JIAF entry", True)
    if extra_in_jiaf:
        check_rule("All JIAF adm2 exist in county_boundary", False, f"({len(extra_in_jiaf)} extra)")
    else:
        check_rule("All JIAF adm2 exist in county_boundary", True)

print("\nBusiness profiling complete.")

BUSINESS PROFILING

--- population_estimates_2024_clean ---

--- jiaf_south_sudan_2026_clean ---

--- health_facilities ---
  facility_name not null

--- education_facilities ---
  adm2_pcode not null

--- Admin Coverage ---
  All adm2 in county_boundary have population estimate (79 missing, e.g. ['SS0711', 'SS0103', 'SS0712'])
  All population adm2 exist in county_boundary
  All adm2 in county_boundary have JIAF entry (79 missing)
  All JIAF adm2 exist in county_boundary

Business profiling complete.
